# Prototipo de fachadas — Revisión del banco local de imágenes

Ejecuta las celdas **en orden** para evaluar con CLIP las fotos de la carpeta `banco_img/` y visualizar el diagnóstico.

## Antes de empezar

1. Crea una carpeta en Google Drive (por ejemplo `MIA/Act_5_VC_12999817`).
2. Sube ahí estos archivos: `2_evaluate_facades.py`, `3_app_visor.py`, `requirements.txt`, la carpeta `banco_img/` (73 fotos) y `manifiesto_banco.csv`.
3. En Colab activa la GPU: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4**.
4. Ejecuta las celdas siguientes en orden.

In [ ]:
# 1) Montar Google Drive y ubicarse en la carpeta del proyecto
from google.colab import drive
drive.mount('/content/drive')

import os
RUTA = '/content/drive/MyDrive/MIA/Act_5_VC_12999817'  # <-- ajusta a tu carpeta
os.chdir(RUTA)
print('Directorio de trabajo:', os.getcwd())
!ls -la

In [ ]:
# 2) Instalar dependencias (torch ya viene con GPU en Colab)
!pip install -q requests transformers safetensors accelerate huggingface_hub pillow numpy pandas folium streamlit streamlit-folium
print('Dependencias listas')

In [ ]:
# 3) Verificar GPU
import torch
print('torch', torch.__version__)
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Sin GPU. Ve a Entorno de ejecución -> Cambiar tipo -> GPU T4 y vuelve a ejecutar.')

In [ ]:
# 4) Evaluar las fachadas del banco local con CLIP
# Genera diagnostico_facades.csv con latitud, longitud, categoria, score y ruta_imagen.
!python 2_evaluate_facades.py --images banco_img --manifest manifiesto_banco.csv --model openai/clip-vit-large-patch14

## Ver el resultado

**Opción A (recomendada):** descarga `resultado.zip` y abre el visor en tu PC (no necesita GPU).

**Opción B:** abre el visor directamente en Colab con un túnel (celda siguiente).

In [ ]:
# 5) Empaquetar y descargar resultados
!zip -r resultado.zip banco_img diagnostico_facades.csv
from google.colab import files
files.download('resultado.zip')
print('En tu PC: descomprime y ejecuta  streamlit run 3_app_visor.py')

## Opción B: visor en Colab con túnel (requiere cuenta gratuita de ngrok)

1. Crea tu token en https://dashboard.ngrok.com/auth.
2. Pégalo en la celda de abajo y ejecuta.

In [ ]:
# 6) Lanzar el visor Streamlit y exponerlo con ngrok
!pip install -q pyngrok
from pyngrok import ngrok
ngrok.set_auth_token('PEGA_AQUI_TU_TOKEN_DE_NGROK')  # <-- tu token

import os, subprocess, time
os.environ['STREAMLIT_SERVER_PORT'] = '8501'
proc = subprocess.Popen(['streamlit', 'run', '3_app_visor.py', '--server.port', '8501', '--server.headless', 'true', '--server.enableCORS', 'false', '--server.enableXsrfProtection', 'false'])
time.sleep(8)
url = ngrok.connect(8501, 'http')
print('Abre el visor en:', url.public_url)